In [1]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [2]:
%pip -q install ultralytics deep-sort-realtime

In [3]:
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# Pretrained object detector
model = YOLO("yolo11n.pt")

# Object tracker with persistent IDs
tracker = DeepSort(max_age=30)

print("YOLO: READY")
print("Deep SORT: READY")
print("Classes:", len(model.names))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


/usr/local/lib/python3.13/dist-packages/deep_sort_realtime/embedder/embedder_pytorch.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


YOLO: READY
Deep SORT: READY
Classes: 80


In [4]:
import urllib.request
import subprocess
import cv2

url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi"

source_path = "/content/vtest.avi"
video_path = "/content/sample_15s.mp4"

# Download video
urllib.request.urlretrieve(url, source_path)

# Extract first 15 seconds
subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", source_path,
    "-t", "15",
    "-an",
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    video_path
], check=True)

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Failed to load sample video")

print("VIDEO READY")
print("Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Resolution:",
      int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
      "x",
      int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))

cap.release()

VIDEO READY
Frames: 150
FPS: 10.0
Resolution: 768 x 576


In [5]:

import cv2
import time
import subprocess
from IPython.display import Video, display
from deep_sort_realtime.deepsort_tracker import DeepSort

# Reset tracker at the start of each video
tracker = DeepSort(
    max_age=30,
    n_init=3,
    embedder_gpu=True
)

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Cannot open the uploaded video.")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 25.0

source_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
source_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Resize large videos to reduce processing cost
scale = min(1.0, 640 / source_width)
width = max(2, int(source_width * scale) // 2 * 2)
height = max(2, int(source_height * scale) // 2 * 2)

raw_output = "/content/tracking_raw.mp4"
final_output = "/content/tracking_output.mp4"

writer = cv2.VideoWriter(
    raw_output,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Cannot create output video.")

frame_count = 0
start = time.perf_counter()

while True:
    success, frame = cap.read()

    if not success:
        break

    frame = cv2.resize(frame, (width, height))

    # 1. YOLO detection
    result = model.predict(
        frame,
        device=0,
        imgsz=640,
        conf=0.35,
        verbose=False
    )[0]

    detections = []

    for box in result.boxes:
        x1, y1, x2, y2 = map(
            float, box.xyxy[0].tolist()
        )

        confidence = float(box.conf[0])
        class_id = int(box.cls[0])
        class_name = model.names[class_id]

        box_width = x2 - x1
        box_height = y2 - y1

        if box_width <= 0 or box_height <= 0:
            continue

        detections.append((
            [x1, y1, box_width, box_height],
            confidence,
            class_name
        ))

    # 2. Deep SORT tracking
    tracks = tracker.update_tracks(
        detections,
        frame=frame
    )

    # 3. Draw bounding boxes and tracking IDs
    for track in tracks:
        if not track.is_confirmed():
            continue

        if track.time_since_update > 0:
            continue

        x1, y1, x2, y2 = map(
            int, track.to_ltrb()
        )

        label = (
            f"{track.get_det_class()} "
            f"ID:{track.track_id}"
        )

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            label,
            (x1, max(20, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2
        )

    writer.write(frame)
    frame_count += 1

cap.release()
writer.release()

elapsed = time.perf_counter() - start
processing_fps = frame_count / max(elapsed, 1e-9)

# Convert to browser-friendly H.264
subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", raw_output,
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    final_output
], check=True)

print("PROCESSING COMPLETE")
print("Frames:", frame_count)
print(f"Processing FPS: {processing_fps:.2f}")
print("Output:", final_output)

display(Video(final_output, embed=True, width=640))


PROCESSING COMPLETE
Frames: 150
Processing FPS: 10.92
Output: /content/tracking_output.mp4


In [6]:
from google.colab import files

files.download("/content/tracking_output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import urllib.request
import cv2

url = (
    "https://raw.githubusercontent.com/intel-iot-devkit/"
    "sample-videos/master/"
    "person-bicycle-car-detection.mp4"
)

video_path = "/content/traffic_test.mp4"

urllib.request.urlretrieve(url, video_path)

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Video download failed")

fps = cap.get(cv2.CAP_PROP_FPS)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("VIDEO READY")
print("Frames:", frames)
print("FPS:", fps)
print("Duration:", round(frames / fps, 2) if fps > 0 else "Unknown")

cap.release()

VIDEO READY
Frames: 647
FPS: 12.0
Duration: 53.92


In [8]:

import cv2
import time
import subprocess
from IPython.display import Video, display
from deep_sort_realtime.deepsort_tracker import DeepSort

# Reset tracker at the start of each video
tracker = DeepSort(
    max_age=30,
    n_init=3,
    embedder_gpu=True
)

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Cannot open the uploaded video.")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 25.0

source_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
source_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Resize large videos to reduce processing cost
scale = min(1.0, 640 / source_width)
width = max(2, int(source_width * scale) // 2 * 2)
height = max(2, int(source_height * scale) // 2 * 2)

raw_output = "/content/tracking_raw.mp4"
final_output = "/content/tracking_output.mp4"

writer = cv2.VideoWriter(
    raw_output,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Cannot create output video.")

frame_count = 0
start = time.perf_counter()

while True:
    success, frame = cap.read()

    if not success:
        break

    frame = cv2.resize(frame, (width, height))

    # 1. YOLO detection
    result = model.predict(
        frame,
        device=0,
        imgsz=640,
        conf=0.35,
        verbose=False
    )[0]

    detections = []

    for box in result.boxes:
        x1, y1, x2, y2 = map(
            float, box.xyxy[0].tolist()
        )

        confidence = float(box.conf[0])
        class_id = int(box.cls[0])
        class_name = model.names[class_id]

        box_width = x2 - x1
        box_height = y2 - y1

        if box_width <= 0 or box_height <= 0:
            continue

        detections.append((
            [x1, y1, box_width, box_height],
            confidence,
            class_name
        ))

    # 2. Deep SORT tracking
    tracks = tracker.update_tracks(
        detections,
        frame=frame
    )

    # 3. Draw bounding boxes and tracking IDs
    for track in tracks:
        if not track.is_confirmed():
            continue

        if track.time_since_update > 0:
            continue

        x1, y1, x2, y2 = map(
            int, track.to_ltrb()
        )

        label = (
            f"{track.get_det_class()} "
            f"ID:{track.track_id}"
        )

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            label,
            (x1, max(20, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2
        )

    writer.write(frame)
    frame_count += 1

cap.release()
writer.release()

elapsed = time.perf_counter() - start
processing_fps = frame_count / max(elapsed, 1e-9)

# Convert to browser-friendly H.264
subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", raw_output,
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    final_output
], check=True)

print("PROCESSING COMPLETE")
print("Frames:", frame_count)
print(f"Processing FPS: {processing_fps:.2f}")
print("Output:", final_output)

display(Video(final_output, embed=True, width=640))


PROCESSING COMPLETE
Frames: 647
Processing FPS: 51.64
Output: /content/tracking_output.mp4


In [9]:
from google.colab import files

files.download("/content/tracking_output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>